In [ ]:
import os
import gurobipy as gp
os.environ["GRB_LICENSE_FILE"] = "/path/to/gurobi.lic"
gp.Model()


In [ ]:
import pickle
from pathlib import Path
from typing import Optional
import dask.array as da
import numpy as np
import pandas as pd
import scipy.ndimage as ndi
import seaborn as sns
import zarr

from tifffile import imread, imwrite
from tqdm import tqdm
from IPython.display import display

from numpy.typing import ArrayLike
from ultrack import track, to_tracks_layer, tracks_to_zarr
from ultrack.utils import estimate_parameters_from_labels, labels_to_contours
from ultrack.config import MainConfig

from ultrack.imgproc import normalize
from ultrack.imgproc.segmentation import reconstruction_by_dilation, Cellpose
from ultrack.utils.array import array_apply, create_zarr
from ultrack.utils.cuda import import_module, to_cpu, torch_default_device

from rich import print
from pyift.shortestpath import watershed_from_minima
from skimage.segmentation import relabel_sequential
from skimage.filters import threshold_otsu
import skimage.morphology as morph
from cellpose import models
from cellpose import core, io
import torch

try:
    import cupy as xp
    import cupyx.scipy.ndimage as xndi
except ImportError:
    import numpy as xp
    import scipy.ndimage as xndi


In [ ]:
img_path = Path("/path/to/image.tif")  # (T, C, Y, X)
save_dir = "/path/to/save/plots"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
foreground_paths = [f"/path/for/zarr/foreground_ch{i}.zarr" for i in range(2)] #specify number of channels e.g 2

normalized_path = "/path/to/zarr/normalized.zarr"
cellpose_path = "/path/to/zarr/cellpose_labels.zarr"
watershed_path = "/path/to/zarr/ws_labels.zarr"

nuclear_channel_index = 0 #segmentation channel
chunks = None

In [ ]:
def remove_background(image, sigma=15.0):
    image = xp.asarray(image)
    seeds = xndi.gaussian_filter(image, sigma=sigma)
    background = reconstruction_by_dilation(seeds, image, iterations=100)
    foreground = xp.maximum(image, background) - background
    return foreground.get()

def watershed_segm(frame, aux_labels, min_area=250):
    import numpy as np
    import cupy
    import cupyx.scipy.ndimage as cndi
    from skimage.filters import threshold_otsu
    from skimage.segmentation import watershed
    from skimage.morphology import remove_small_objects

    frame = cupy.asarray(frame)
    aux_labels = cupy.asarray(aux_labels)

    # Gaussian filter
    frame = frame.astype(cupy.float32)
    frame = cndi.gaussian_filter(frame, 3.0)

    # Threshold and detection
    frame_cpu = frame.get()
    aux_labels_cpu = aux_labels.get()
    det = frame_cpu > (threshold_otsu(frame_cpu) * 0.75)
    det = np.logical_or(det, aux_labels_cpu > 0)

    # Conversion for distance transform
    det_gpu = cupy.asarray(det)
    distance = cndi.distance_transform_edt(det_gpu)

    # Watershed
    markers = aux_labels_cpu.astype(np.int32)
    labels = watershed(-distance.get(), markers, mask=det)
    labels = remove_small_objects(labels, min_size=min_area)

    return labels.astype(np.int32)

def gpu_find_boundaries(label_frame: np.ndarray, mode: str = 'outer') -> np.ndarray:
    boundaries = np.zeros_like(label_frame, dtype=bool)
    padded = np.pad(label_frame, pad_width=1, mode='edge')
    center = padded[1:-1, 1:-1]
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            neighbor = padded[1 + di:1 + di + label_frame.shape[0], 1 + dj:1 + dj + label_frame.shape[1]]
            boundaries |= (center != neighbor)
    return boundaries.astype(np.float32)

def gpu_labels_to_contours(labels, sigma=None):
    import cupy
    import cupyx.scipy.ndimage as cndi
    if not isinstance(labels, (list, tuple)):
        labels = [labels]
    shape = labels[0].shape
    for lb in labels:
        if lb.shape != shape:
            raise ValueError("Mismatched label shapes")
    foreground = np.zeros(shape, dtype=bool)
    contours = np.zeros(shape, dtype=np.float32)
    for t in range(shape[0]):
        fg_frame = np.zeros(shape[1:], dtype=bool)
        ct_frame = np.zeros(shape[1:], dtype=np.float32)
        for lb in labels:
            lb_frame = lb[t]
            fg_frame |= (lb_frame > 0)
            ct_frame += gpu_find_boundaries(lb_frame)
        ct_frame /= len(labels)
        if sigma is not None:
            ct_frame = cupy.asarray(ct_frame)
            ct_frame = cndi.gaussian_filter(ct_frame, sigma)
            max_val = ct_frame.max()
            if max_val > 0:
                ct_frame /= max_val
        foreground[t] = fg_frame.get() if hasattr(fg_frame, "get") else fg_frame
        contours[t] = ct_frame.get() if hasattr(ct_frame, "get") else ct_frame
    return foreground, contours

In [ ]:
imgs = imread(img_path)
T, C, Y, X = imgs.shape
print(f"Image size (T, C, Y, X) = {imgs.shape}")

In [ ]:
# Background removal for all channels
foregrounds = []
for c in range(C):
    ch_img = imgs[:, c]
    chunks = (1, *ch_img.shape[1:])
    out_path = foreground_paths[c]
    fg = create_zarr(ch_img.shape, ch_img.dtype, out_path, chunks=chunks, overwrite=True)
    array_apply(ch_img, out_array=fg, func=remove_background, sigma=15.0, axis=0)
    foregrounds.append(fg)

In [ ]:
# Normalize and segment nuclear channel
nuclear_fg = foregrounds[nuclear_channel_index]
normalized = create_zarr(nuclear_fg.shape, np.float16, normalized_path, chunks=chunks, overwrite=True)
array_apply(nuclear_fg, out_array=normalized, func=normalize, gamma=0.5, axis=0)

use_gpu = torch.cuda.is_available() if hasattr(torch.cuda, 'is_available') else torch.backends.mps.is_available()
model = models.CellposeModel(gpu=use_gpu, pretrained_model='cpsam')
cellpose_labels = create_zarr(normalized.shape, np.uint16, cellpose_path, chunks=chunks, overwrite=True)

def apply_cellpose(img):
    masks, flows, styles = model.eval(img)
    return masks

array_apply(normalized, out_array=cellpose_labels, func=apply_cellpose, axis=0)

print("Using GPU:", use_gpu)

In [ ]:
# Watershed segmentation
ws_labels = create_zarr(normalized.shape, np.int32, watershed_path, chunks=chunks, overwrite=True)
array_apply(normalized, cellpose_labels, out_array=ws_labels, func=watershed_segm, min_area=250, axis=0)

In [ ]:
# Contour extraction
cp_np = np.asarray(cellpose_labels)
ws_np = np.asarray(ws_labels)
foreground, contours = gpu_labels_to_contours([cp_np, ws_np], sigma=5.0)

In [ ]:
#Review background substraction, noramalization, cellpose segmentation, watershed segmentation and contour extraction

import matplotlib.pyplot as plt
import numpy as np

# Choose a frame index and a channel index
frame_idx = 0
channel_idx = 0

first_frame = imgs[frame_idx, channel_idx] 

# Prepare the segmentation results for that frame
frames = {
    "Raw Image": first_frame,
    "Foreground": foreground[frame_idx],
    "Normalize": normalized[frame_idx],
    "Contours": contours[frame_idx],
    "Cellpose Labels": cellpose_labels[frame_idx],
    "Watershed Labels": ws_labels[frame_idx],
}

fig_seg, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

cmaps = ["gray", "gray", "gray", "gray", "Paired", "Paired"]

for ax, (title, frame), cmap in zip(axes, frames.items(), cmaps):
    ax.imshow(frame, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

axes[4].contour(contours[frame_idx], colors='white', linewidths=0.3)

plt.tight_layout()
plt.show()


In [ ]:
#Calculate displacement between two frames
import numpy as np
from scipy.ndimage import center_of_mass
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt

def get_centroids(label_frame):
    """
    Compute centroids for each label (cell) in a segmentation mask.
    
    Parameters
    ----------
    label_frame : ndarray
        2D segmentation mask (labels > 0 represent cells).
        
    Returns
    -------
    dict
        Dictionary mapping label IDs to centroid coordinates.
    """
    labels = np.unique(label_frame)
    labels = labels[labels > 0]
    centroids = {}
    for label in labels:
        centroids[label] = center_of_mass(label_frame == label)
    return centroids

# choose two succesive frames to compare
labels0 = ws_np[20]
labels1 = ws_np[21]

centroids0 = get_centroids(labels0)
centroids1 = get_centroids(labels1)

coords0 = np.array(list(centroids0.values()))
coords1 = np.array(list(centroids1.values()))

dist_matrix = cdist(coords0, coords1)

row_ind, col_ind = linear_sum_assignment(dist_matrix)

matched_distances = dist_matrix[row_ind, col_ind]

max_disp = np.percentile(matched_distances, 95)
max_disp_exact = matched_distances.max()
average_disp = np.mean(matched_distances)
perc_disp = np.percentile(matched_distances, 25)

print(f"Estimated max displacement (95th percentile): {max_disp:.2f} pixels")
print(f"Maximum displacement among matched cells: {max_disp_exact:.2f} pixels")
print(f"Average displacement among matched cells: {average_disp:.2f} pixels")
print(f"Estimated max displacement (25th percentile): {perc_disp:.2f} pixels")

fig_disp, ax = plt.subplots(figsize=(8, 6))
ax.hist(matched_distances, bins=30, edgecolor='black')
ax.set_xlabel("Cell displacement (pixels)")
ax.set_ylabel("Frequency")
ax.set_title("Histogram of cell displacements between frames")

fig.savefig("/path/to/save/displacement_histogram.svg", dpi=600, bbox_inches='tight')

In [ ]:
#Nuclei area distribution

# If using watershed labels, load them from Zarr:
#ws_zarr = zarr.open("/path/to/ws_labels.zarr", mode="r")
#ws_labels_np = ws_zarr[:]  # This is a NumPy array with shape (time, height, width)
#params_df = estimate_parameters_from_labels(ws_labels_np, is_timelapse=True)

# If you prefer to use Cellpose labels instead, do:
cellpose_zarr = zarr.open("/path/to/cellpose_labels.zarr", mode="r")
cellpose_labels_np = cellpose_zarr[:]
params_df = estimate_parameters_from_labels(cp_np, is_timelapse=True)

fig_area, ax_area = plt.subplots(figsize=(8, 6))

params_df["area"].plot(kind="hist", bins=100, title="Area histogram", ax=ax_area)
ax_area.set_xlabel("Area (pixel units)")
ax_area.set_ylabel("Frequency")

In [ ]:
#Ultrack parameters

config = MainConfig()
n_workers = 8

#Change based on your data
config.segmentation_config.n_workers = n_workers
config.segmentation_config.min_area = 50
config.segmentation_config.max_area = 950
config.segmentation_config.min_frontier = 0.01 
config.segmentation_config.threshold = 0.5

config.linking_config.max_distance = 40
config.linking_config.n_workers = n_workers
config.linking_config.z_score_threshold = 3.0

# Tracking integer linear programming (ILP) parameters
config.tracking_config.division_weight = -0.01
config.tracking_config.disappear_weight = -2
config.tracking_config.appear_weight = -0.1

# ILP processing window size.
# It reduces memory usage while asserting continuity in the tracks.
config.tracking_config.window_size = 15
config.tracking_config.overlap_size = 3
config.tracking_config.solution_gap = 0.0

# Working directory
config.data_config.working_dir = Path("/path/to/working_dir")

print(config)

In [ ]:
# Run Ultrack
track(config, detection=foreground, edges=contours, images=[normalized], overwrite=True)

# Convert the tracking results to a Pandas DataFrame and lineage graph
tracks_df, lineage_graph = to_tracks_layer(config)

# Convert tracking results to Zarr format
tracking_labels = tracks_to_zarr(config, tracks_df)

In [ ]:
def plot_tracks(tracks_df: pd.DataFrame) -> None:
    centered_df = tracks_df.copy()
    centered_df[["y", "x"]] = centered_df.groupby("track_id", as_index=False)[["y", "x"]].transform(lambda x: x - x.iloc[0])
    assert (centered_df[centered_df["t"] == 0][["y", "x"]] == 0).all().all()
    sns.lineplot(data=centered_df, x="x", y="y", hue="track_id", legend=False, alpha=0.5, sort=False, estimator=None)
    return centered_df

In [ ]:
#Plot cell tracks for quality control

centered_df = plot_tracks(tracks_df)
fig_tracks_2d = plt.gcf()

In [ ]:
#3d plot centered
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_tracks_3d(tracks_df: pd.DataFrame):
    """
    Center tracks at their initial position and plot them in 3D including time dimension.
    
    Parameters
    ----------
    time_tracks : pd.DataFrame
        Tracks dataframe sorted by `track_id` and `t`.
    
    Returns
    -------
    tuple
        A tuple containing:
         - centered_df: The centered DataFrame.
         - fig: The matplotlib figure handle for the 3D plot.
    """
    centered_df = tracks_df.copy()
    centered_df[["y", "x"]] = centered_df.groupby(
        "track_id",
        as_index=False,
    )[["y", "x"]].transform(lambda x: x - x.iloc[0])
    
    
    assert (centered_df[centered_df["t"] == 0][["y", "x"]] == 0).all().all()
    
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    
    track_ids = centered_df["track_id"].unique()
    
    
    palette = sns.color_palette("husl", len(track_ids))
    
    
    for i, track_id in enumerate(track_ids):
        track_data = centered_df[centered_df["track_id"] == track_id]
        ax.plot(track_data["x"], track_data["y"], track_data["t"],
                color=palette[i], alpha=0.7)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Time')
    ax.set_title('3D Visualization of Tracks')
    
    
    ax.legend()
    
    
    return centered_df, fig

# Example of how to call and save the figure later:
if __name__ == "__main__":
    # centered_df, fig_tracks_3d = plot_tracks_3d(time_tracks)
    # fig_tracks_3d.savefig("/your/save/directory/tracks_3d.svg", dpi=600, bbox_inches='tight')
    pass



In [ ]:
#plot 3d tracks

centered_df, fig_tracks_3d = plot_tracks_3d(tracks_df)
fig_tracks_3d.savefig(os.path.join(save_dir, "figure_name.svg"), dpi=600, bbox_inches='tight')

In [ ]:
#3d plot not centered

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_tracks_3d_nocenter(tracks_df: pd.DataFrame):
    """
    Plot tracks in 3D including the time dimension without centering them.
    
    Parameters
    ----------
    time_tracks : pd.DataFrame
        Tracks dataframe sorted by `track_id` and `t`. Expected to contain at least the columns:
        "track_id", "x", "y", and "t".
    
    Returns
    -------
    tuple
        A tuple containing:
         - original_df: The original DataFrame.
         - fig: The matplotlib figure handle for the 3D plot.
    """
    
    original_df = tracks_df.copy()
    
   
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    
    track_ids = original_df["track_id"].unique()
    
    
    palette = sns.color_palette("husl", len(track_ids))
    
    
    for i, track_id in enumerate(track_ids):
        track_data = original_df[original_df["track_id"] == track_id]
        ax.plot(track_data["x"], track_data["y"], track_data["t"],
                color=palette[i], alpha=0.7)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Time')
    ax.set_title('3D Visualization of Tracks (Not Centered)')
    
    ax.legend()
    
    return original_df, fig




In [ ]:
original_df, fig_tracks_3d_no_center = plot_tracks_3d_nocenter(tracks_df)
plt.show()
fig_tracks_3d_no_center.savefig("path/to/save/image.svg", dpi=600, bbox_inches='tight')

In [ ]:
#Save output files

results_dir = Path("path/to/save/folder")
tracks_df.to_csv(results_dir / "tracks_file.csv", index=False)
imwrite(results_dir / "path/to/save/ultrack_tracking_labels.tif", tracking_labels)
imwrite(results_dir / "path/to/save/cellpose_labels.tif", cellpose_labels)
print(f"Saved results into {results_dir}")

In [ ]:
# Extract properties for each channel
from ultrack.imgproc import tracks_properties
import pandas as pd

all_dfs = []

for i, fg in enumerate(foregrounds):
    
    df = tracks_properties(tracking_labels, image=fg)

    
    df["channel"] = i

    
    df = df.sort_values(by=["track_id", "t"]).reset_index(drop=True)
    df["time_min"] = df["t"] * 10

    
    cols = ["track_id", "t", "time_min", "channel"] + [col for col in df.columns if col not in ["track_id", "t", "time_min", "channel"]]
    df = df[cols]

    all_dfs.append(df)


combined_df = pd.concat(all_dfs, ignore_index=True)


id_cols = ["track_id", "t", "time_min"]
value_cols = [col for col in combined_df.columns if col not in id_cols + ["channel"]]


pivoted_df = combined_df.pivot_table(
    index=id_cols,
    columns="channel",
    values=value_cols
)


pivoted_df.columns = [f"{col}_ch{ch}" for col, ch in pivoted_df.columns]
pivoted_df = pivoted_df.reset_index()


pivoted_df.to_csv(results_dir / "properties.csv", index=False)